In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Лабораторная 1

Загрузка датасета

In [39]:
df = pd.read_csv("../data/raw/penguins.csv")

In [40]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB


Разделение датасета

In [41]:
X = df.drop(columns=["species"])
y = df["species"]

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [43]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category", "bool", "str"]).columns.tolist()

In [44]:
print("Числовые:", num_cols)
print("Категориальные:", cat_cols)

Числовые: ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
Категориальные: ['island', 'sex']


Создаем свой трансформер по заданию

In [45]:
class RareCategoryReplacer(BaseEstimator, TransformerMixin):
    """
    Заменяет категории, встречающиеся реже, чем threshold (доля),
    на строку 'Other'. Работает только с категориальными столбцами.

    Параметры
    ----------
    threshold : float, default=0.05
        Минимальная доля встречаемости категории. Всё, что реже записывается как 'Other'.
    """

    def __init__(self, threshold=0.05):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.frequent_categories_ = {}

        n = len(X)
        for col in X.columns:
            counts = X[col].value_counts(normalize=True, dropna=False)
            keep = counts[counts >= self.threshold].index
            self.frequent_categories_[col] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col in X.columns:
            if col in self.frequent_categories_:
                keep = self.frequent_categories_[col]
                # всё, что не в keep, и не NaN -> 'Other'
                mask = ~X[col].isin(keep) & X[col].notna()
                X.loc[mask, col] = "Other"
        return X

Собираем ColumnTransformer

In [46]:
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

In [47]:
cat_pipeline = Pipeline(steps=[
    ("rare", RareCategoryReplacer(threshold=0.05)),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

In [48]:
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
])

Собираем пайплайн

In [49]:
full_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

In [50]:
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)
y_proba = full_pipeline.predict_proba(X_test)

In [51]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba, multi_class="ovr"))

Accuracy: 1.0
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        30
   Chinstrap       1.00      1.00      1.00        14
      Gentoo       1.00      1.00      1.00        25

    accuracy                           1.00        69
   macro avg       1.00      1.00      1.00        69
weighted avg       1.00      1.00      1.00        69

ROC-AUC: 1.0


Сохраняем модель

In [52]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

In [53]:
model_path = models_dir / "penguins_logreg.joblib"
joblib.dump(full_pipeline, model_path)

['..\\models\\penguins_logreg.joblib']

In [54]:
print("Модель сохранена:", model_path.resolve())
print("Размер:", model_path.stat().st_size, "байт")

Модель сохранена: C:\Users\malvia\Documents\GitHub\adhm_2\models\penguins_logreg.joblib
Размер: 4970 байт
